# Load Harmonic Key Transitions Into Neo4j

## Load useful libraries

In [ ]:
import os
import pandas as pd

In [ ]:
from music_production_and_DJ_tools.theory.western.scales.pitch_class_scales.chromatic import pitch_class_chromatic_scale_camelot_system
from music_production_and_DJ_tools.theory.western.scales.pitch_class_scales.chromatic import pitch_class_chromatic_scale_opposite_of_camelot_system

In [ ]:
from music_production_and_DJ_tools.dj.harmonic_mixing.camelot_wheel import CamelotWheel

In [ ]:
from python_tools_and_shortcuts.databases.neo4j.Neo4jInterface import Neo4jInterface

## User settings

In [ ]:
neo4j_password = os.environ['NEO4J_PASSWORD']

## Assemble all the chromatics we want to include

In [ ]:
list_chromatic_scales = [
    pitch_class_chromatic_scale_camelot_system,
    pitch_class_chromatic_scale_opposite_of_camelot_system,
]

## Generate the Camelot wheel for harmonic mixing

In [ ]:
c = CamelotWheel(list_chromatic_scales)

## QA

In [ ]:
c.df_initial_wheel.head()

In [ ]:
c.df_initial_wheel_stacked.head(3)

In [ ]:
c.df_key_nodes.head(3)

In [ ]:
c.df_key_nodes.tail(3)

In [ ]:
c.df_relationships

## Connect to Neo4j

In [ ]:
nj = Neo4jInterface(neo4j_password)

## Clear the keys existing in the database

In [ ]:
nj.drop_everything_by_node_type('MUSIC_KEY')

## Load nodes

In [ ]:
list_dict_nodes = c.df_key_nodes.to_dict(orient = 'records')

In [ ]:
def batch_key_merge(tx, batch):
    query = """
    UNWIND $batch AS row
    MERGE (n1:MUSIC_KEY {name: row.key, key: row.key, camelot_key: row.camelot_key})
    """
    tx.run(query, batch=batch)

In [ ]:
nj.batch_it(batch_key_merge, list_dict_nodes)

## Load relationships

#### Transitions down a 5th

In [ ]:
list_dict_relationships = c.df_relationships[['key', 'down_key']].drop_duplicates().to_dict(orient = 'records')

In [ ]:
def batch_merge_down(tx, batch):
    query = """
    UNWIND $batch AS row
    MERGE (n1:MUSIC_KEY {name: row.key})
    MERGE (n2:MUSIC_KEY {name: row.down_key})
    MERGE (n1)-[r:HAS_HARMONIC_TRANSITION_TO]->(n2)
    """
    tx.run(query, batch=batch)

In [ ]:
nj.batch_it(batch_merge_down, list_dict_relationships)

#### Transitions up a 5th

In [ ]:
list_dict_relationships = c.df_relationships[['key', 'up_key']].drop_duplicates().to_dict(orient = 'records')

In [ ]:
def batch_merge_up(tx, batch):
    query = """
    UNWIND $batch AS row
    MERGE (n1:MUSIC_KEY {name: row.key})
    MERGE (n2:MUSIC_KEY {name: row.up_key})
    MERGE (n1)-[r:HAS_HARMONIC_TRANSITION_TO]->(n2)
    """
    tx.run(query, batch=batch)

In [ ]:
nj.batch_it(batch_merge_up, list_dict_relationships)

#### Mode transitions

In [ ]:
list_dict_relationships = c.df_relationships[['key', 'mode_shift_key']].drop_duplicates().to_dict(orient = 'records')

In [ ]:
def batch_merge_mode_shift(tx, batch):
    query = """
    UNWIND $batch AS row
    MERGE (n1:MUSIC_KEY {name: row.key})
    MERGE (n2:MUSIC_KEY {name: row.mode_shift_key})
    MERGE (n1)-[r:HAS_HARMONIC_TRANSITION_TO]->(n2)
    """
    tx.run(query, batch=batch)

In [ ]:
nj.batch_it(batch_merge_mode_shift, list_dict_relationships)

#### Transitions to the same key (including enharmonics)

In [ ]:
list_dict_relationships = c.df_relationships[['key', 'same_shift_key']].drop_duplicates().to_dict(orient = 'records')

In [ ]:
def batch_merge_same_shift(tx, batch):
    query = """
    UNWIND $batch AS row
    MERGE (n1:MUSIC_KEY {name: row.key})
    MERGE (n2:MUSIC_KEY {name: row.same_shift_key})
    MERGE (n1)-[r:HAS_HARMONIC_TRANSITION_TO]->(n2)
    """
    tx.run(query, batch=batch)

In [ ]:
nj.batch_it(batch_merge_same_shift, list_dict_relationships)